# datalayer

## Overview

The Hera datalayer serves as a datalake, utilizing MongoDB to store metadata describing data properties and linking to corresponding data files. Data generated from measurements or simulations is often extensive. Therefore, it's common practice to store the data on disk while retaining only metadata in the database. Each database entry, referred to as a 'document', represents a single piece of data (such as pandas, dask, xarray, etc.) associated with a specific project. 

Managing the documents is performed using the [project](./Project.ipynb) interface. Alternatively the user can 
use a [lowlevel](./lowlevel.ipynb) interface. The Project interface simplifies the use of datalayer by 
allowing the user to specify the project name once, when the Project object is constructed,rather than requiring it every time a document is manipulated.

As MongoDB is a NoSQL database, metadata for each data piece is presented in JSON format, creating a hierarchical data structure. Users can query data pieces based on the JSON structure, specifying queries using the MongoEngine query language.

In basic usage, users add data documents to the database by providing metadata and specifying the data path. Upon addition, users define the data format. This definition allows users to employ the getData() method, enabling them to load the data while considering the data format for suitable loading strategies.


Each record denotes a piece of data from various sources. For simplicity, we define three distinct types of data documents:

<table>
  <tr>
      <th>Document kind</th>
    <th>Description</th>
  </tr>
  <tr>
    <td><b>Measurements</b></td>
    <td>Document references data from measurements or external data</td>
  </tr>
  <tr>
      <td><b>Simulations</b></td>
      <td>Document references data from simulations</td>      
    </tr>
  <tr>
      <td><b>Cache</b></td>
    <td>Document references data after computation (either measurement or simulation data)</td>
  </tr>
</table>

Each document holds the following properties: 
<table>
  <tr>
      <th>Property name</th>
    <th>Description</th>
  </tr>
  <tr>
    <td><b>projectName</b></td>
    <td>The term used to identify the project to which a particular piece of information belongs.</td>
  </tr>
  <tr>
      <td><b>type</b></td>
      <td>A user-defined string to identify the data</td>      
    </tr>
  <tr>
    <td><b>dataFormat</b></td>
      <td>The <a href="#datatypes">format</a> of the data the document stores</td>
  </tr>
  <tr>
    <td><b>resource</b></td>
    <td>The resource that the document holds (for example the path to the file)</td>
  </tr>

</table>

![Hera-DB](../Hera-DB.png)

Each user has their own database that stores documents, but it is also possible to access
other databases.

## Setup database connection 

The connection to the database can be managed through command line interface (CLI) or manually. 

The default connection is the **linux user name**.
It is possible to add other database connections. Instructions for adding additional connections will be provided in the future. 

### Command line interface (CLI)

The command line allows the user to list, add or remove database connections. 

#### Listing the database connections. 

Database connections can be listed using the following CLI command:

<div class="alert alert-success" role="alert">    
    >> hera-project db list
</div>    

#### Adding a new database connection 

Database connection can be added with the follow command:

<div class="alert alert-success" role="alert">    
    >> hera-project db create connectionName --username USERNAME --password PASSWORD --IP IP --databaseName DATABASENAME
</div>    

Where 

- **connectionName** : The name of the connection. 
- **USERNAME**  : The username of the mongo database. 
- **PASSWORD**  : The password to the mongo database. 
- **IP**  : The IP of the database server. 
- **DATABASENAME** : The database name. 

<div class="alert alert-block alert-info">
<b>Example: </b> 
    
```
>> hera-project db create myConnection --username usr --password usr_abc --IP 127.0.0.1 --databaseName usr
```
    
Results in adding a connection myConnection with the user name usr, password usr_abc, IP 127.0.0.1 and database name is usr. 
</div>        

#### Removing a database connection 

Database connection can be removed using:
<div class="alert alert-success" role="alert">    
    >> hera-project db remove connectionName 
</div>    

As a result, the CLI will print the removed connection data as JSON. 

<div class="alert alert-block alert-info">
<b>Example: </b> 
    
```
>> hera-project db remove myConnection 
```
    
Results in removing the connection myConnection and printing 
```javascript
{
    "dbIP": "127.0.0.1",
    "dbName": "usr",
    "password": "usr_abc",
    "username": "usr"
}

```
</div>        



### Manual management 

The database is configured in the file 
`$HOME/.pyhera/config.json`. 

The structure of the config.json is:

```javascript
{
        <connection name 1> : {
            "dbIP": "DB IP",
            "dbName": "...",
            "password": "..." ,
            "username": "..."
        },
            .
            .
            .
}
```

Where 
- **username**: The username of the mongo database. 
- **password**: The password to the mongo database. 
- **dbName**: The database name. 
- **dbIP** : The IP of the database server. 

<a id="datatypes"></a>
## Datatypes 

The format of the data items are: 

- **STRING**: Represents data stored as a string.
- **TIME**: Represents time-based data.
- **CSV_PANDAS**: Represents data stored in CSV format and compatible with the pandas library.
- **HDF**: Represents data stored in HDF format.
- **NETCDF_XARRAY**: Represents data stored in NetCDF format and compatible with the xarray library.
- **JSON_DICT**: Represents data stored in JSON format as a dictionary.
- **JSON_PANDAS**: Represents data stored in JSON format and compatible with the pandas library.
- **JSON_GEOPANDAS**: Represents data stored in JSON format and compatible with the geopandas library.
- **GEOPANDAS**: Represents geographical data compatible with the geopandas library.
- **PARQUET**: Represents data stored in Parquet format.
- **IMAGE**: Represents image data.
- **PICKLE**: Represents data serialized using Python's pickle module.
- **DICT**: Represents data stored as a dictionary.

Accessing the datatype is obtained through the datatype object

# 10-minute tutorial

This tutorial demonstrate how to store and retrieve data from the database
with the default connection (the username of the linux system). 

First, lets create some mock-up data that we can store in the DB.   

In [ ]:
import pandas
import numpy
from scipy.stats import norm

x = numpy.linspace(norm.ppf(0.01), norm.ppf(0.99), 100)

dataset1 = pandas.DataFrame(dict(x=x,y=norm.pdf(x,loc=0,scale=1)))
dataset2 = pandas.DataFrame(dict(x=x,y=norm.pdf(x,loc=0,scale=0.5)))
dataset3 = pandas.DataFrame(dict(x=x,y=norm.pdf(x,loc=0.5,scale=0.5)))

print(dataset1.head())

Now that we have data, we can save it. We would like to keep the connection between the data and the parameters that generated it. 
So that:

- **dataset1** is characterized by loc=0 and scale = 1 
- **dataset2** is characterized by loc=0 and scale = 0.5 
- **dataset3** is characterized by loc=0.5 and scale = 0.5

Therefore, we will save the loc and scale in the metadata.

Before we add the data to the DB, we need to save it to the disk

In [ ]:
import os 

# getting the work directory
workingdir = os.path.join(os.path.abspath(os.getcwd()),"examples","datalayer")
os.makedirs(workingdir, exist_ok=True)


dataset1File = os.path.join(workingdir,"dataset1.parquet")
dataset2File = os.path.join(workingdir,"dataset2.parquet")
dataset3File = os.path.join(workingdir,"dataset3.parquet")

Now we can save the dataset. We choose parquet for convinience, but it can be any other format.

In [ ]:
dataset1.to_parquet(dataset1File,engine='pyarrow',compression='GZIP')
dataset2.to_parquet(dataset2File,engine='pyarrow',compression='GZIP')
dataset3.to_parquet(dataset3File,engine='pyarrow',compression='GZIP')

When we save the data to the database we need to define a project
and specify the project name. 

To do so, we need to import the project and create it with its name. 
For this example we will use the name `ExampleProject`.

In [ ]:
from hera.datalayer import Project

projectName = "ExampleProject"

proj = Project(projectName=projectName)

## Adding data items to the project 

Next, we add the documents to the database. To do this, we must specify the 'type' of the documents. This type is user-defined and enables the user to query all documents of this type.

For this example, we will add the data as a Measurement data. 

In [ ]:
proj.addMeasurementsDocument(type="Distribution",
                             dataFormat=proj.datatypes.PARQUET,
                             resource=dataset1File,
                             desc=dict(loc=0,scale=1));

proj.addMeasurementsDocument(type="Distribution",
                             dataFormat=proj.datatypes.PARQUET,
                             resource=dataset2File,
                             desc=dict(loc=0,scale=0.5));

proj.addMeasurementsDocument(type="Distribution",
                             dataFormat=proj.datatypes.PARQUET,
                             resource=dataset3File,
                             desc=dict(loc=0.5,scale=0.5));

## Getting the data

### Getting one record back
Now we will query the database for all the records in which loc=0 and scale=1.

In [ ]:
List1 = proj.getMeasurementsDocuments(loc=0,scale=1)

print(f"The number of documents obtained from the query {len(List1)} ")
item0 = List1[0]

Note that for consistency the query always returns a list.

The description of the record that matched the query is

In [ ]:
import json 

print("The description of dataset 1")
print(json.dumps(item0.desc, indent=4, sort_keys=True))

Now, we will extract the data.

Using the getData on item0 will retrieve the data 

In [ ]:
dataset1FromDB = item0.getData()

Since the data is parquet, the library automatically returns a dask.DataFrame, where 
the data is not loaded until it is computed. 

Alternatively, we can pass the usePandas flag. This flag is used only 
when the datatype is PARQUET. 

In [ ]:
dataset1FromDB = item0.getData(usePandas=True)

print(dataset1FromDB)

### Getting multiple records back

The getMeasurementsDocuments returns all the records that match the criteria. 

Now, lets get all the records where loc=0

In [ ]:
List2 = proj.getMeasurementsDocuments(loc=0)

print(f"The number of documents obtained from the query {len(List2)} ")

As another example, let's retrieve all documents of the type 'Distribution'.

In [ ]:
List3 = proj.getMeasurementsDocuments(type='Distribution')

print(f"The number of documents obtained from the query {len(List3)} ")

## Updating the data.
The hera system holds the name of the file on the disk and loads the data from it. Therefore, if the datafile on the disk is overwitten, then the data of the record is changed

Lets multiply dataset1 by factor 2. The file name is saved in the resource attribute.

Note that if we just update the data and not the metadata, then we can use the resource property to 
save the new file. 

In [ ]:
dataset1['y'] *=2
dataset1FileName = item0.resource
dataset1.to_parquet(dataset1FileName,engine='pyarrow',compression='GZIP')

In [ ]:
dataset1FromDB = item0.getData().compute()
print(dataset1FromDB)

## Updating the metadata.

Lets assume we want to add another property to the first record. To so we will update item0

In [ ]:
item0.desc['new_attribute'] = "some data"
item0.save();

Lets requery the database to see what is the data there. 

In [ ]:
item0_fromdb = proj.getMeasurementsDocuments(loc=0,scale=1)[0]
print(json.dumps(item0_fromdb.desc, indent=4, sort_keys=True))

## Deleting the metadata entry.

We delete the metadata records similarly to the way we add them

The following will delete one record. The simplest method is to erase the document object. 

In [ ]:
item0.delete()

Lets query the database again to see if the record was deleted. 

<div class="alert alert-block alert-warning">
Note that the file on the disk is not deleted by deleting the record in the DB. 
</div>

In [ ]:
List1 = proj.getMeasurementsDocuments(loc=0,scale=1)

print(f"The number of documents obtained from the query {len(List1)} ")

Another option is to delete the records using the Project interface. 

In [ ]:
deletedList1 = proj.deleteMeasurementsDocuments(loc=0)

Lets list all the data records that we deleted

In [ ]:
for doc in deletedList1:
    print(json.dumps(doc, indent=4, sort_keys=True))

## Deleting the data on the disk 

Now we can erase the file from the disk. It is saved in the resource property

In [ ]:
import shutil

for doc in deletedList1:
    if os.path.isfile(doc['resource']):
          os.remove(doc['resource'])
    else:
        shutil.rmtree(doc['resource'])

## Delete all the metadata records 

A simple way to delete all the records (be careful)

In [ ]:
[x.delete() for x in proj.getMeasurementsDocuments(type="Distribution")]